# Week 5 - Q7: Is lead time seasonal?
**Question.** Delivery lead time averages 8.06 days and is uniform by region. Is there a seasonal pattern by month or year, and is there enough structure to forecast?

**Approach.** The unit of analysis is the **Delivered order** (18,033 rows, one lead time each). Lead time = `delivery_date - order_date` (`days_to_deliver`). Cancelled orders also carry a delivery date (an artefact found in Q3), so they stay out of every measure. Months are tested three ways (Kruskal-Wallis, a seasonal wave fitted to the 72 months, and a comparison of the 72 monthly means with sampling noise), and every other cut is scanned together with a permutation test so that looking at many cuts does not manufacture a finding. Order volume and revenue are checked for the same pattern, and a hold-out test on 2024 says whether a seasonal index would help a forecast.

**Not the same as supplier lead time.** Customer delivery lead time (order date to delivery date, 0 to 15 days) is a different measure from supplier lead time (10.5 to 28.5 days, Q8). They are never mixed here. Built on the corrected Week 2 merged datasets. Seed 42, 5,000 resamples.

In [1]:
import pandas as pd, numpy as np, os, itertools, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
def find_dir(env_var, marker, rels):
    """Environment variable first, then this folder and every parent folder, using the repository layout."""
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    return None
DATA_DIR = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged"])
assert DATA_DIR is not None, "Could not find the merged datasets. Run from inside the repository, put a merged/ folder next to the notebook, or set the CADETX_MERGED environment variable to the folder."
OUT=Path("q7_outputs"); OUT.mkdir(exist_ok=True); CH=Path("q7_charts"); CH.mkdir(exist_ok=True)
SEED=42; NBOOT=5000; NPERM=5000
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def wilson(x,n,z=1.959964):
    p=x/n; d=1+z*z/n; c=(p+z*z/(2*n))/d; h=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d; return c-h,c+h
def holm(p):
    p=np.asarray(p,float); m=len(p); o=np.argsort(p); adj=np.empty(m); run=0
    for r,i in enumerate(o): run=max(run,(m-r)*p[i]); adj[i]=min(1,run)
    return adj
MON=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
print("Merged datasets:",DATA_DIR.resolve())

Merged datasets: <repository>/week-02/reports/week_02/merged


## Part 0. Load and integrity checks

In [2]:
lines=pd.read_csv(DATA_DIR/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date","delivery_date"])
check("sales lines",130402,len(lines),len(lines)==130402)
o=lines.drop_duplicates("so_id").copy()
check("unique orders",20000,len(o),len(o)==20000)
D=o[o.order_status=="Delivered"].copy()
check("delivered orders (Week 3)",18033,len(D),len(D)==18033)
check("cancelled orders (Week 3)",1967,int((o.order_status=="Cancelled").sum()),int((o.order_status=="Cancelled").sum())==1967)
check("lead time is one value per order (same on every line)",0,int((lines.groupby("so_id").days_to_deliver.nunique()>1).sum()),(lines.groupby("so_id").days_to_deliver.nunique()>1).sum()==0)
dd=(D.delivery_date-D.order_date).dt.days
check("days_to_deliver equals delivery date minus order date on every delivered order",0,int((dd!=D.days_to_deliver).sum()),(dd!=D.days_to_deliver).sum()==0)
rev_all=lines.loc[lines.order_status=="Delivered","line_total"].sum()
check("net revenue of delivered orders is Rs 22.96B (Weeks 3 and 4)",22.96,round(rev_all/1e9,2),round(rev_all/1e9,2)==22.96)
x=D.days_to_deliver.astype(float)
check("mean lead time is 8.06 days (Week 3)",8.06,round(x.mean(),2),round(x.mean(),2)==8.06)
check("median lead time is 8 days",8.0,float(x.median()),float(x.median())==8.0)
check("lead time runs from 0 to 15 days",(0,15),(int(x.min()),int(x.max())),(x.min(),x.max())==(0,15))
canc=o[o.order_status=="Cancelled"].days_to_deliver
check("cancelled orders carry the same lead time (8.05 days) and are excluded",8.05,round(canc.mean(),2),round(canc.mean(),2)==8.05)
D["month"]=D.order_date.dt.month; D["year"]=D.order_date.dt.year; D["ym"]=D.order_date.dt.to_period("M"); D["dow"]=D.order_date.dt.dayofweek
print(f"{len(D):,} delivered orders; lead time mean {x.mean():.2f}, median {x.median():.0f}, sd {x.std():.2f}, range {int(x.min())} to {int(x.max())} days; cancelled orders (excluded) average {canc.mean():.2f} days")

18,033 delivered orders; lead time mean 8.06, median 8, sd 4.30, range 0 to 15 days; cancelled orders (excluded) average 8.05 days


## Part 1. What the lead-time distribution looks like
Before asking about months, look at the shape. A seasonal pattern needs a lead time that responds to something; a flat spread of 1 to 15 days responds to nothing.

In [3]:
dist=x.value_counts().sort_index().rename_axis("days").reset_index(name="orders"); dist["share_pct"]=dist.orders/len(D)*100; dist.to_csv(OUT/"q7_01_lead_time_distribution.csv",index=False)
c1=x[x>=1].value_counts().sort_index()
chi_u,p_u=stats.chisquare(c1)
check("lead times of 1 to 15 days are consistent with a uniform spread (chi-square p > 0.05)",">0.05",round(p_u,3),p_u>0.05)
print(f"1 to 15 days: each value holds {c1.min():,} to {c1.max():,} orders ({c1.min()/len(D)*100:.1f}% to {c1.max()/len(D)*100:.1f}%); uniform chi-square p = {p_u:.3f}; same-day deliveries: {int((x==0).sum())}")
print(f"Share delivered in 12+ days: {(x>=12).mean()*100:.1f}%")

1 to 15 days: each value holds 1,133 to 1,277 orders (6.3% to 7.1%); uniform chi-square p = 0.125; same-day deliveries: 6
Share delivered in 12+ days: 26.9%


## Part 2. Month of the year
The 12 calendar months of the order date, each with its mean, median, spread and a bootstrap interval. Test: Kruskal-Wallis (rank based, no normality assumption) with epsilon-squared, then one-way ANOVA eta-squared and Levene (does the spread change by month?).

In [4]:
rng=np.random.default_rng(SEED)
rows=[]
for m,g in D.groupby("month"):
    v=g.days_to_deliver.values.astype(float); bs=rng.choice(v,(NBOOT,len(v))).mean(1); late=int((v>=12).sum()); lo,hi=wilson(late,len(v))
    rows.append(dict(month=m,name=MON[m-1],orders=len(v),mean_days=v.mean(),ci_low=np.percentile(bs,2.5),ci_high=np.percentile(bs,97.5),median_days=np.median(v),sd_days=v.std(ddof=1),share_12plus_pct=late/len(v)*100,share_ci_low=lo*100,share_ci_high=hi*100))
M=pd.DataFrame(rows); M["index_vs_overall"]=M.mean_days/x.mean(); M.to_csv(OUT/"q7_02_lead_time_by_month.csv",index=False)
groups=[g.days_to_deliver.values for _,g in D.groupby("month")]
H,p_kw=stats.kruskal(*groups); k=len(groups); eps2=max(0.0,(H-k+1)/(len(D)-k))
F,p_an=stats.f_oneway(*groups); eta2=sum(len(a)*(a.mean()-x.mean())**2 for a in groups)/((x-x.mean())**2).sum()
lev,p_lev=stats.levene(*groups)
spread=M.mean_days.max()-M.mean_days.min()
check("month of year does not change lead time (Kruskal-Wallis p > 0.05)",">0.05",round(p_kw,3),p_kw>0.05)
check("month effect size is negligible (epsilon-squared < 0.01)","<0.01",round(eps2,5),eps2<0.01)
check("highest and lowest monthly mean are under half a day apart","<0.5",round(spread,3),spread<0.5)
check("spread of lead time does not change by month (Levene p > 0.05)",">0.05",round(p_lev,3),p_lev>0.05)
pairs=[(a,b,stats.mannwhitneyu(groups[a],groups[b]).pvalue) for a,b in itertools.combinations(range(12),2)]
PW=pd.DataFrame(pairs,columns=["i","j","p_raw"]); PW["month_a"]=[MON[i] for i in PW.i]; PW["month_b"]=[MON[j] for j in PW.j]; PW["p_holm"]=holm(PW.p_raw)
PW["mean_gap_days"]=[M.mean_days[i]-M.mean_days[j] for i,j in zip(PW.i,PW.j)]; PW.drop(columns=["i","j"]).to_csv(OUT/"q7_03_pairwise_months.csv",index=False)
check("no pair of months differs after Holm adjustment over 66 pairs (smallest adjusted p > 0.05)",">0.05",round(PW.p_holm.min(),3),PW.p_holm.min()>0.05)
sd=x.std(); nmin=M.orders.min(); mde=(1.959964+0.841621)*sd*np.sqrt(1/nmin+1/(len(D)-nmin)); mde_typ=(1.959964+0.841621)*sd*np.sqrt(1/M.orders.median()+1/(len(D)-M.orders.median()))
check("a typical month would show a gap of under half a day at 80% power","<0.5",round(mde_typ,2),mde_typ<0.5)
pd.DataFrame([dict(test="Kruskal-Wallis",statistic=H,p=p_kw,effect_size=eps2,effect_name="epsilon_squared"),dict(test="One-way ANOVA",statistic=F,p=p_an,effect_size=eta2,effect_name="eta_squared"),dict(test="Levene (spread)",statistic=lev,p=p_lev,effect_size=np.nan,effect_name=""),dict(test="Smallest pairwise p after Holm (66 pairs)",statistic=np.nan,p=PW.p_holm.min(),effect_size=np.nan,effect_name=""),dict(test="Detectable gap, typical month (80% power, days)",statistic=np.nan,p=np.nan,effect_size=mde_typ,effect_name="days"),dict(test="Detectable gap, smallest month (days)",statistic=np.nan,p=np.nan,effect_size=(1.959964+0.841621)*sd*np.sqrt(1/nmin+1/(len(D)-nmin)),effect_name="days")]).to_csv(OUT/"q7_04_month_tests.csv",index=False)
print(M[["name","orders","mean_days","ci_low","ci_high","median_days","share_12plus_pct"]].round(2).to_string(index=False))
print(f"Kruskal-Wallis p = {p_kw:.3f}, epsilon-squared = {eps2:.5f}; ANOVA p = {p_an:.3f}, eta-squared = {eta2:.5f}; Levene p = {p_lev:.3f}")
print(f"Monthly means run {M.mean_days.min():.2f} ({M.name[M.mean_days.idxmin()]}) to {M.mean_days.max():.2f} ({M.name[M.mean_days.idxmax()]}) days, a spread of {spread:.2f}. A typical month would show a gap of {mde_typ:.2f} days; the smallest ({M.name[M.orders.idxmin()]}, {nmin:,} orders) {(1.959964+0.841621)*sd*np.sqrt(1/nmin+1/(len(D)-nmin)):.2f} days.")

name  orders  mean_days  ci_low  ci_high  median_days  share_12plus_pct
 Jan    1531       8.13    7.91     8.34          8.0             27.82
 Feb    1358       8.20    7.97     8.42          8.0             28.28
 Mar    1509       7.92    7.70     8.13          8.0             24.92
 Apr    1441       8.16    7.93     8.38          8.0             28.38
 May    1551       8.04    7.82     8.25          8.0             26.63
 Jun    1437       8.10    7.88     8.32          8.0             27.42
 Jul    1595       8.03    7.82     8.24          8.0             26.71
 Aug    1554       7.97    7.76     8.19          8.0             25.55
 Sep    1504       8.12    7.90     8.33          8.0             27.33
 Oct    1506       8.02    7.81     8.23          8.0             25.76
 Nov    1550       8.11    7.90     8.34          8.0             27.55
 Dec    1497       7.98    7.76     8.19          8.0             26.19
Kruskal-Wallis p = 0.852, epsilon-squared = 0.00000; ANOVA p = 0

## Part 3. The 72 months as a time series
A seasonal pattern repeats every 12 months. Three checks use the 72 monthly means: a fitted yearly wave (sine and cosine), the correlation of each month with the month 12 steps earlier, and whether the 72 means vary more than sampling noise alone would make them.

In [5]:
ms=D.groupby("ym").days_to_deliver.agg(["mean","count"]); ms["month"]=ms.index.month; ms["year"]=ms.index.year
t=np.arange(len(ms)); check("series has 72 months",72,len(ms),len(ms)==72)
def harmonic(y,mon):
    X=np.column_stack([np.ones(len(y)),np.sin(2*np.pi*mon/12),np.cos(2*np.pi*mon/12)]); b=np.linalg.lstsq(X,y,rcond=None)[0]
    r1=((y-X@b)**2).sum(); r0=((y-y.mean())**2).sum(); F=((r0-r1)/2)/(r1/(len(y)-3)); return F,stats.f.sf(F,2,len(y)-3),float(np.hypot(b[1],b[2]))
Fh,p_h,amp=harmonic(ms["mean"].values,ms.month.values)
check("no yearly wave in the 72 monthly means (harmonic F test p > 0.05)",">0.05",round(p_h,3),p_h>0.05)
lag1=pd.Series(ms["mean"].values).autocorr(1); lag12=pd.Series(ms["mean"].values).autocorr(12)
rho_t,p_t=stats.spearmanr(t,ms["mean"].values)
z=(ms["mean"]-x.mean())/(x.std()/np.sqrt(ms["count"])); chi_d=float((z**2).sum()); p_d=stats.chi2.sf(chi_d,71)
check("the 72 monthly means vary no more than sampling noise (chi-square p > 0.05)",">0.05",round(p_d,3),p_d>0.05)
yr=D.groupby("year").days_to_deliver.agg(["count","mean","median","std"]).reset_index(); yr.to_csv(OUT/"q7_05_lead_time_by_year.csv",index=False)
H_y,p_y=stats.kruskal(*[g.days_to_deliver.values for _,g in D.groupby("year")])
check("year does not change lead time (Kruskal-Wallis p > 0.05)",">0.05",round(p_y,3),p_y>0.05)
ms.reset_index().assign(ym=lambda d:d.ym.astype(str)).to_csv(OUT/"q7_06_monthly_series.csv",index=False)
pd.DataFrame([dict(test="Yearly wave (harmonic F)",statistic=Fh,p=p_h,detail=f"amplitude {amp:.3f} days ({amp/x.mean()*100:.2f}% of the mean)"),dict(test="72 means vs sampling noise (chi-square, 71 df)",statistic=chi_d,p=p_d,detail=f"sd of monthly means {ms['mean'].std():.3f} vs {x.std()/np.sqrt(ms['count'].mean()):.3f} expected from noise"),dict(test="Trend over 72 months (Spearman)",statistic=rho_t,p=p_t,detail=""),dict(test="Autocorrelation lag 1 / lag 12",statistic=lag1,p=np.nan,detail=f"lag 12 = {lag12:.3f}"),dict(test="Years (Kruskal-Wallis)",statistic=H_y,p=p_y,detail="")]).to_csv(OUT/"q7_07_series_tests.csv",index=False)
print(yr.round(3).to_string(index=False))
print(f"Yearly wave: F = {Fh:.2f}, p = {p_h:.3f}, amplitude {amp:.3f} days. Lag-1 {lag1:.2f}, lag-12 {lag12:.2f}. Trend rho = {rho_t:.2f} (p = {p_t:.2f}).")
print(f"Sampling noise alone would give monthly means with sd {x.std()/np.sqrt(ms['count'].mean()):.3f}; the observed sd is {ms['mean'].std():.3f} (chi-square p = {p_d:.2f}).")

 year  count  mean  median   std
 2019   3031 8.093     8.0 4.291
 2020   3038 8.078     8.0 4.279
 2021   3038 8.042     8.0 4.372
 2022   2948 8.081     8.0 4.291
 2023   3022 8.081     8.0 4.286
 2024   2956 8.000     8.0 4.294
Yearly wave: F = 0.19, p = 0.830, amplitude 0.026 days. Lag-1 0.07, lag-12 -0.02. Trend rho = -0.11 (p = 0.35).
Sampling noise alone would give monthly means with sd 0.272; the observed sd is 0.256 (chi-square p = 0.79).


## Part 4. Every other cut, scanned together
The sprint plan says lead time is uniform by region. Ten cuts are tested (month, year, customer region, branch region, channel, category, customer type, industry, payment terms, warehouse type). A permutation scan shuffles the lead times 5,000 times, repeats all ten tests and records the smallest p-value, which answers: would a scan this strong appear by chance?

In [6]:
cuts=["month","year","customer_region","branch_region","sales_channel","category","customer_type","industry_segment","payment_terms","warehouse_type"]
codes={c:pd.factorize(D[c])[0] for c in cuts}; v=x.values
def anova_p(v):
    ps=[]; sst=((v-v.mean())**2).sum()
    for c in cuts:
        kk=codes[c].max()+1; ss=np.bincount(codes[c],weights=v,minlength=kk); n=np.bincount(codes[c],minlength=kk)
        ssb=(ss**2/n).sum()-v.sum()**2/len(v); Fv=(ssb/(kk-1))/((sst-ssb)/(len(v)-kk)); ps.append(stats.f.sf(Fv,kk-1,len(v)-kk))
    return np.array(ps)
p_obs=anova_p(v); rngp=np.random.default_rng(SEED); mins=np.array([anova_p(rngp.permutation(v)).min() for _ in range(NPERM)]); p_fw=float((mins<=p_obs.min()).mean())
rows=[]
for c,pa in zip(cuts,p_obs):
    gg=[g.days_to_deliver.values for _,g in D.groupby(c)]; Hc,pk=stats.kruskal(*gg); kk=len(gg)
    rows.append(dict(cut=c,groups=kk,p_anova=pa,p_kruskal=pk,epsilon_squared=max(0.0,(Hc-kk+1)/(len(D)-kk)),mean_low=min(g.mean() for g in gg),mean_high=max(g.mean() for g in gg)))
SC=pd.DataFrame(rows); SC["p_holm"]=holm(SC.p_kruskal); SC["gap_days"]=SC.mean_high-SC.mean_low; SC.to_csv(OUT/"q7_08_all_cuts.csv",index=False)
reg=D.groupby("customer_region").days_to_deliver.agg(["count","mean","std"]).reset_index(); reg.to_csv(OUT/"q7_09_by_customer_region.csv",index=False)
check("ten cuts scanned and every effect size is negligible (largest epsilon-squared < 0.01)","<0.01",round(SC.epsilon_squared.max(),5),SC.epsilon_squared.max()<0.01)
check("nothing is significant after Holm over the ten cuts (smallest adjusted p > 0.05)",">0.05",round(SC.p_holm.min(),3),SC.p_holm.min()>0.05)
check("whole-scan permutation: the smallest p-value is no stronger than chance (family-wise p > 0.05)",">0.05",round(p_fw,3),p_fw>0.05)
dw=stats.kruskal(*[g.days_to_deliver.values for _,g in D.groupby("dow")]).pvalue
check("weekday of the order does not change lead time (p > 0.05)",">0.05",round(dw,3),dw>0.05)
rv=stats.spearmanr(D.total_order_value,x)[0]
check("order value is unrelated to lead time (|Spearman rho| < 0.05)","<0.05",round(abs(rv),4),abs(rv)<0.05)
print(SC.round(4).to_string(index=False))
print(f"Smallest raw p: {SC.loc[SC.p_kruskal.idxmin(),'cut']} (p = {SC.p_kruskal.min():.4f}); after Holm {SC.p_holm.min():.3f}; whole-scan permutation p = {p_fw:.3f}")
print(reg.round(3).to_string(index=False))

             cut  groups  p_anova  p_kruskal  epsilon_squared  mean_low  mean_high  p_holm  gap_days
           month      12   0.8514     0.8515           0.0000    7.9178     8.2010  1.0000    0.2832
            year       6   0.9627     0.9643           0.0000    7.9997     8.0930  1.0000    0.0934
 customer_region       6   0.0079     0.0081           0.0006    7.5567     8.1623  0.0805    0.6057
   branch_region       4   0.6650     0.6668           0.0000    7.9825     8.1027  1.0000    0.1202
   sales_channel       4   0.3018     0.2922           0.0000    8.0018     8.1621  1.0000    0.1603
        category      15   0.5133     0.5211           0.0000    7.8266     8.3074  1.0000    0.4809
   customer_type       5   0.5454     0.5379           0.0000    7.9881     8.1504  1.0000    0.1623
industry_segment       6   0.7576     0.7616           0.0000    8.0006     8.1344  1.0000    0.1338
   payment_terms       5   0.5299     0.5137           0.0000    8.0008     8.1592  1.0000 

## Part 5. Do order volume and revenue show the seasonal pattern?
Volume = Delivered orders by order month; revenue = net `line_total` of those orders. Raw monthly counts differ partly because months have different numbers of days, so volume is tested against a days-proportional expectation, and both series are also tested for a yearly wave.

In [7]:
Dl=lines[lines.order_status=="Delivered"].copy(); Dl["ym"]=Dl.order_date.dt.to_period("M")
S=pd.DataFrame({"revenue":Dl.groupby("ym").line_total.sum(),"orders":D.groupby("ym").size(),"lead_time":D.groupby("ym").days_to_deliver.mean()})
S["month"]=S.index.month; S["year"]=S.index.year; S["days_in_month"]=S.index.days_in_month
check("revenue series reconciles to Rs 22.96B",22.96,round(S.revenue.sum()/1e9,2),round(S.revenue.sum()/1e9,2)==22.96)
IDX=S.groupby("month")[["lead_time","orders","revenue"]].mean()/S[["lead_time","orders","revenue"]].mean(); IDX.index=MON
IDX.columns=["lead_time_index","orders_index","revenue_index"]; IDX["orders_per_day_index"]=(S.orders/S.days_in_month).groupby(S.month).mean().values/(S.orders/S.days_in_month).mean()
IDX["revenue_per_day_index"]=(S.revenue/S.days_in_month).groupby(S.month).mean().values/(S.revenue/S.days_in_month).mean()
IDX.reset_index(names="month").to_csv(OUT/"q7_10_seasonal_index.csv",index=False)
cn=S.groupby("month").orders.sum(); expd=S.groupby("month").days_in_month.sum()/S.days_in_month.sum()*cn.sum(); chi_c,p_c=stats.chisquare(cn,expd)
raw_kw={c:stats.kruskal(*[g.values for _,g in S[c].groupby(S.month)]).pvalue for c in ["orders","revenue","lead_time"]}
harm={c:harmonic(S[c].values,S.month.values) for c in ["orders","revenue","lead_time"]}
check("order counts follow month length: no month pattern beyond the number of days (chi-square p > 0.05)",">0.05",round(p_c,3),p_c>0.05)
check("no yearly wave in monthly orders (harmonic p > 0.05)",">0.05",round(harm["orders"][1],3),harm["orders"][1]>0.05)
check("no yearly wave in monthly revenue (harmonic p > 0.05)",">0.05",round(harm["revenue"][1],3),harm["revenue"][1]>0.05)
pd.DataFrame([dict(series=c,raw_monthly_kruskal_p=raw_kw[c],yearly_wave_p=harm[c][1],wave_amplitude_pct_of_mean=harm[c][2]/S[c].mean()*100,coefficient_of_variation_pct=S[c].std()/S[c].mean()*100,trend_spearman=stats.spearmanr(np.arange(72),S[c])[0]) for c in ["lead_time","orders","revenue"]]).to_csv(OUT/"q7_11_volume_revenue_seasonality.csv",index=False)
print(IDX.round(3).to_string())
print(f"Raw monthly tests: orders p = {raw_kw['orders']:.3f}, revenue p = {raw_kw['revenue']:.3f} (February is short). Orders against days-in-month: chi-square p = {p_c:.2f}.")
print(f"Yearly wave p: orders {harm['orders'][1]:.3f} (amplitude {harm['orders'][2]/S.orders.mean()*100:.1f}%), revenue {harm['revenue'][1]:.3f} ({harm['revenue'][2]/S.revenue.mean()*100:.1f}%), lead time {harm['lead_time'][1]:.2f} ({harm['lead_time'][2]/S.lead_time.mean()*100:.1f}%)")

     lead_time_index  orders_index  revenue_index  orders_per_day_index  revenue_per_day_index
Jan            1.008         1.019          1.037                 1.001                  1.018
Feb            1.017         0.904          0.902                 0.972                  0.970
Mar            0.982         1.004          1.014                 0.986                  0.996
Apr            1.012         0.959          0.978                 0.973                  0.992
May            0.996         1.032          1.014                 1.014                  0.996
Jun            1.004         0.956          0.944                 0.971                  0.958
Jul            0.996         1.061          1.061                 1.043                  1.042
Aug            0.989         1.034          1.058                 1.016                  1.039
Sep            1.005         1.001          1.005                 1.016                  1.020
Oct            0.994         1.002          1.021 

## Part 6. Is there enough structure to forecast?
A seasonal index is only worth using if it forecasts better than ignoring the season. Hold-out test: build each forecast from 2019 to 2023 and score it on the 12 months of 2024 (mean absolute percentage error, lower is better). Methods: the plain 2019 to 2023 mean; the mean times a month-of-year index; the same month of 2023 (seasonal naive); and the last 12 months' mean.

In [8]:
rows=[]
for c,label in [("lead_time","Lead time (days)"),("orders","Delivered orders"),("revenue","Net revenue")]:
    tr=S[S.year<2024]; te=S[S.year==2024]; mean=tr[c].mean(); ix=tr.groupby("month")[c].mean()/mean
    f={"Plain mean (2019 to 2023)":np.full(12,mean),"Mean x month index":te.month.map(ix).values*mean,"Same month last year (2023)":S[S.year==2023].set_index("month")[c].reindex(te.month).values,"Mean of last 12 months":np.full(12,tr[c].iloc[-12:].mean())}
    for name,fc in f.items(): rows.append(dict(series=label,method=name,mape_pct=float((abs(te[c].values-fc)/te[c].values).mean()*100)))
HO=pd.DataFrame(rows); HO.to_csv(OUT/"q7_12_holdout_2024.csv",index=False)
for label in HO.series.unique():
    h=HO[HO.series==label].set_index("method").mape_pct
    check(f"{label}: the month index does not forecast 2024 better than the plain mean",f">={h['Plain mean (2019 to 2023)']:.2f}",round(h['Mean x month index'],2),h['Mean x month index']>=h['Plain mean (2019 to 2023)'])
print(HO.pivot(index="method",columns="series",values="mape_pct").round(2).to_string())

series                       Delivered orders  Lead time (days)  Net revenue
method                                                                      
Mean of last 12 months                   6.88              2.18         7.02
Mean x month index                       6.88              2.38         8.24
Plain mean (2019 to 2023)                6.83              2.16         7.49
Same month last year (2023)              8.36              2.70         9.36


## Part 7. Charts

In [9]:
def style(ax):
    ax.set_facecolor(SURF)
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    for s in ["left","bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2,labelsize=9); ax.grid(axis="y",color=GRID,lw=0.7); ax.set_axisbelow(True)
fig,ax=plt.subplots(figsize=(8.2,4.2),facecolor=SURF); style(ax)
ax.bar(dist.days,dist.orders,color=BLUE,width=0.78); ax.axhline(c1.mean(),color=INK2,lw=1,ls="--"); ax.text(15.4,c1.mean()+25,f"uniform level {c1.mean():,.0f}",ha="right",color=INK2,fontsize=9)
ax.set_xticks(range(0,16)); ax.set_xlabel("Lead time (days)",color=INK2); ax.set_ylabel("Delivered orders",color=INK2); ax.set_title("Lead time is spread evenly from 1 to 15 days",loc="left",color=INK,fontsize=12,fontweight="bold")
plt.tight_layout(); plt.savefig(CH/"q7_01_lead_time_distribution.png",dpi=150,facecolor=SURF); plt.close()
fig,ax=plt.subplots(figsize=(8.2,4.2),facecolor=SURF); style(ax)
ax.errorbar(M.month,M.mean_days,yerr=[M.mean_days-M.ci_low,M.ci_high-M.mean_days],fmt="o",color=BLUE,ecolor=BLUE,capsize=3,ms=6)
ax.axhline(x.mean(),color=ORANGE,lw=1.5); ax.text(0.6,x.mean()+0.03,f"overall {x.mean():.2f}",color=ORANGE,fontsize=9,ha="left")
ax.set_xticks(range(1,13)); ax.set_xticklabels(MON); ax.set_ylim(7.2,8.9); ax.set_ylabel("Mean lead time (days), 95% interval",color=INK2)
ax.set_title(f"Month of order does not change lead time (p = {p_kw:.2f})",loc="left",color=INK,fontsize=12,fontweight="bold")
plt.tight_layout(); plt.savefig(CH/"q7_02_lead_time_by_month.png",dpi=150,facecolor=SURF); plt.close()
fig,axes=plt.subplots(1,3,figsize=(11,3.9),facecolor=SURF,sharey=True)
for ax,(c,lab,col) in zip(axes,[("lead_time_index","Lead time",BLUE),("orders_index","Orders",AQUA),("revenue_index","Revenue",ORANGE)]):
    style(ax); ax.bar(range(1,13),IDX[c].values,color=col,width=0.7); ax.axhline(1,color=INK2,lw=1); ax.set_xticks(range(1,13)); ax.set_xticklabels([m[0] for m in MON]); ax.set_title(lab,loc="left",color=INK,fontsize=11,fontweight="bold"); ax.set_ylim(0.8,1.12)
axes[0].set_ylabel("Index (1.00 = average month)",color=INK2)
fig.suptitle("Month-of-year index: lead time is flat; orders and revenue vary, but not beyond chance",x=0.01,ha="left",color=INK,fontsize=12,fontweight="bold")
plt.tight_layout(rect=[0,0,1,0.93]); plt.savefig(CH/"q7_03_seasonal_index.png",dpi=150,facecolor=SURF); plt.close()
print("charts saved:",sorted(p.name for p in CH.glob("*.png")))

charts saved: ['q7_01_lead_time_distribution.png', 'q7_02_lead_time_by_month.png', 'q7_03_seasonal_index.png']


## Part 8. Findings

In [10]:
print(f"- Lead time is not seasonal. By month of order the means run {M.mean_days.min():.2f} to {M.mean_days.max():.2f} days (Kruskal-Wallis p = {p_kw:.2f}, epsilon-squared {eps2:.4f}); no pair of months differs after adjustment (smallest adjusted p = {PW.p_holm.min():.2f}); no yearly wave (p = {p_h:.2f}); no trend across 2019 to 2024 (year p = {p_y:.2f}).")
print(f"- Lead time is a flat spread of 1 to 15 days (uniformity p = {p_u:.2f}) that shows no meaningful response to month, year, channel, category, customer type, order value or weekday. Ten cuts scanned together: the smallest raw p is {SC.p_kruskal.min():.3f} ({SC.loc[SC.p_kruskal.idxmin(),'cut']}; Central and North East customers are about 0.3 to 0.6 days quicker, with only about 400 orders each); it is {SC.p_holm.min():.2f} after Holm and {p_fw:.2f} in the whole-scan permutation test, with effect size epsilon-squared {SC.epsilon_squared.max():.4f}. That is a weak lead near the threshold, not a finding, and it is not seasonal.")
print(f"- Order volume and revenue: monthly counts follow month length (p = {p_c:.2f} against days); no yearly wave in orders (p = {harm['orders'][1]:.2f}) or revenue (p = {harm['revenue'][1]:.2f}). The raw monthly tests (p = {raw_kw['orders']:.2f} and {raw_kw['revenue']:.2f}) come mostly from short February.")
print(f"- Forecast readiness: a month-of-year index does not beat a plain average on 2024 for lead time, orders or revenue. There is no reliable seasonal structure to forecast from in this data (orders show a wave of about 3% at p = {harm['orders'][1]:.2f}, too weak to improve a forecast); use averages with the stated ranges.")
print("- Caveat: lead time looks generated (even spread, almost no response to anything). Whether delivery_date is the real delivery or a promised date is a question for the data owner.")

- Lead time is not seasonal. By month of order the means run 7.92 to 8.20 days (Kruskal-Wallis p = 0.85, epsilon-squared 0.0000); no pair of months differs after adjustment (smallest adjusted p = 1.00); no yearly wave (p = 0.83); no trend across 2019 to 2024 (year p = 0.96).
- Lead time is a flat spread of 1 to 15 days (uniformity p = 0.12) that shows no meaningful response to month, year, channel, category, customer type, order value or weekday. Ten cuts scanned together: the smallest raw p is 0.008 (customer_region; Central and North East customers are about 0.3 to 0.6 days quicker, with only about 400 orders each); it is 0.08 after Holm and 0.07 in the whole-scan permutation test, with effect size epsilon-squared 0.0006. That is a weak lead near the threshold, not a finding, and it is not seasonal.
- Order volume and revenue: monthly counts follow month length (p = 0.37 against days); no yearly wave in orders (p = 0.07) or revenue (p = 0.12). The raw monthly tests (p = 0.02 and 0.04

## Part 9. Questions for the data owner
1. Is `delivery_date` the actual delivery date or a planned date? Real delivery times usually respond to season, product, distance or stock.
2. Why is every lead time between 0 and 15 days with an even spread, including for cancelled orders?
3. Is there a dispatch date, a carrier or a delay reason recorded anywhere, so that delivery performance can be measured against a promise?

## Part 10. Validation summary

In [11]:
V=pd.DataFrame(checks); V.to_csv("validation_report_week05_q7.csv",index=False)
print(f"{(V.Status=='PASS').sum()} of {len(V)} checks pass"); V

34 of 34 checks pass


,Check_Item,Expected,Actual,Status
0,sales lines,130402,130402,PASS
1,unique orders,20000,20000,PASS
2,delivered orders (Week 3),18033,18033,PASS
3,cancelled orders (Week 3),1967,1967,PASS
4,lead time is one value per order (same on ever...,0,0,PASS
5,days_to_deliver equals delivery date minus ord...,0,0,PASS
6,net revenue of delivered orders is Rs 22.96B (...,22.96,22.96,PASS
7,mean lead time is 8.06 days (Week 3),8.06,8.06,PASS
8,median lead time is 8 days,8.0,8.0,PASS
9,lead time runs from 0 to 15 days,"(0, 15)","(0, 15)",PASS
